# 08 — Gate G-1: the end-to-end audit before the final run (W6 of stage 9)

**Purpose.** Before the one final training run, an audit of the whole pipeline: when each input
is known against the issuance time, whether the test split touched any decision, whether the
pipeline reproduces from the raw data, and the two analyses of v6 that train models.

**The gate** (`PROJECT_MASTER.md` §7) asks for nine things: the assumptions verified or
declared, the full test suite green, a manual leakage audit, a test split that touched no
decision, a reproduction from scratch, an external review, the smoke and dry runs, the scope
register, and the tag `g1-frozen`.

**Rules.**
- The reproduction is a consistency check of the corrected pipeline, not a match to the submitted
  report (D-34).
- Nothing is changed after the tag unless it only reads frozen outputs (D-36).

**Documented in** OP-60 to OP-62, `docs/leakage_audit.md` §3, `docs/stage9_plan.md` §20 and §21,
run journal R9-25 to R9-32, errata X55 to X75, tests T-93 to T-103 and the figures S09-34 to
S09-41.

## 1. The manual leakage audit: when is each input known?

In [1]:
import re

import pandas as pd

import json

from solarpv.config.paths import DOCS_DIR, OUTPUTS_DIR, REPO_ROOT

text = (DOCS_DIR / "leakage_audit.md").read_text(encoding="utf-8")
block = text.split("### 3.1")[1].split("### 3.2")[0]
rows = [[c.strip() for c in line.strip("|").split("|")] for line in block.splitlines()
        if re.match(r"^\| \d+ \|", line)]
families = pd.DataFrame(rows, columns=["#", "family", "known at", "code", "test"])
families[["#", "family", "test"]]

,#,family,test
0,1,"Measured lags (`*_lag_{1,2,3,7,14}d`, 7 series)","T-09, T-18 (perturbation after T0, both DST ch..."
1,2,Rolling same-hour means and stds (12),T-09
2,3,Imputation of 1–2,T-34
3,4,D-22 masks of the power series,T-35
4,5,"Clear sky and solar geometry (9 + aoi, azimuth...",T-36
5,6,"Calendar (8 time encodings, day of year, day l...",T-36
6,7,GFS (22 `fcst_gfs_*`),"T-17 (every row, every horizon), T-43, T-44"
7,8,"Everything measured at the target hour, the ta...",T-01
8,9,"Encoder of the sequence models (BiLSTM, three ...","T-02, T-66–T-71, T-77"
9,10,Decoder of the sequence models,"T-02, T-78"


**What it shows.** One row for each of the 18 families of inputs, with the
test that proves its rule: the measured lags and rolling statistics, the imputation, the masks, the
calendar and geometry, the GFS run, the encoders and decoders of the sequence models, PatchTST,
the scalers, the L-09 correction, the references, the selection, the tuning and the k history.

**What it means.** No measured input reads an hour that ends after T0. The only information about
the forecast days comes from deterministic functions of time and from the GFS run initialised on
the previous day, every file of which was uploaded before T0 (checked row by row, T-17). The
original audit was a list of forbidden names; it could not state this.

## 2. The steps before the hourly table (X57)

In [2]:
block = text.split("### 3.2")[1].split("### 3.3")[0]
rows = [[c.strip() for c in line.strip("|").split("|")] for line in block.splitlines()
        if re.match(r"^\| \d+ \|", line)]
pd.DataFrame(rows, columns=["#", "step", "look-ahead", "measured", "reading"])[
    ["#", "step", "reading"]]

,#,step,reading
0,19,Ingest (stage 2),—
1,20,"Clock correction (stage 4, D-19)",the estimates measure the logger's clock (the ...
2,21,"Polled reconstruction (stage 4, F-18)","negligible, and only in the test period; declared"
3,22,Cleaning (stage 5),negligible; declared


**What it shows.** The 4 steps that build the hourly table, which no earlier
audit had examined. Three look at data after T0 when they prepare an hour before it: the clock
correction (a centred median), the polled-source reconstruction (interpolation towards the next
poll) and the cleaning (frozen runs, margins and a centred window).

**What it means.** Each was measured, not assumed (next cell), and declared in
`docs/deviations.md`. The user accepted them (D-35).

In [3]:
audit = OUTPUTS_DIR / "g1_audit" / "audit"
clean = pd.read_csv(audit / "lookahead_clean.csv")
clock = pd.read_csv(audit / "lookahead_clock.csv")
polled = pd.read_csv(audit / "lookahead_polled.csv")
display(clean[["source", "issuance_days", "cells_changed", "within_6h_of_t0", "days_affected",
               "test_days_affected", "method_mismatch_cells"]])
display(clock[["clock", "days", "days_shift_differs", "share_pct", "max_shift_difference_min"]])
polled[["source", "polled_issuance_days", "max_bound_on_last_hour_mean"]]

,source,issuance_days,cells_changed,within_6h_of_t0,days_affected,test_days_affected,method_mismatch_cells
0,s01_inv1_AC_kW,1364,1,0,1,0,0
1,s01_inv2_AC_kW,1364,1,0,1,0,0
2,site_total_AC_kW,1364,1,0,1,0,0
3,irradiance,1364,18,16,4,0,0
4,mod_temp1,1364,15,13,3,0,0
5,mod_temp2,1364,15,13,3,0,0
6,ambient_temp,1364,15,13,3,0,0


,clock,days,days_shift_differs,share_pct,max_shift_difference_min
0,minute_meteo,1369,552,40.32,3
1,minute_inverter,1369,541,39.52,3


,source,polled_issuance_days,max_bound_on_last_hour_mean
0,s01_inv1_AC_kW,94,0.0000
1,s01_inv2_AC_kW,94,0.0000
2,irradiance,94,0.2329
3,mod_temp1,94,0.6796
4,mod_temp2,94,0.6520
5,ambient_temp,94,0.6002


**What it shows.** Cleaning: for every one of the 1,364
issuance days the stage-4 minutes were cleaned up to T0 and with the next day, and the source
values of the 48 hours before T0 compared. 66 of
458,304 cells change, on 4 days at most per
series, none of them on a test day. Clock: a trailing median would change the whole-minute
shift on 40 % of the days, by at most
3 minutes. Polled source: over
94 issuance days, the interpolation towards the next poll
changes the mean of the last source hour by 0.68 at
most (the unit of the series, °C for a temperature).

**What it means.** The clock correction is the only look-ahead of size, and it measures the
logger's clock, not the weather or the plant. The method check (`method_mismatch_cells`,
0) shows that the cleaning with the next day equals the
stage-5 table.

## 3. The test split touched no decision

In [4]:
import json

access = json.loads((audit / "test_access.json").read_text(encoding="utf-8"))
pd.DataFrame(access["per_run"])[["run", "step_records", "test_access_entries", "prediction_files",
                                 "rows_at_or_after_test_start", "last_prediction_hour"]]

,run,step_records,test_access_entries,prediction_files,rows_at_or_after_test_start,last_prediction_hour
0,20261105T195802Z_smoke,2,0,3,0,2024-04-11 20:00:00+00:00
1,20261105T195821Z_dev,2,0,3,0,2024-09-30 23:00:00+00:00
2,20261107T104110Z_smoke,3,0,4,0,2024-04-11 20:00:00+00:00
3,20261107T104151Z_dev,3,0,4,0,2024-09-30 23:00:00+00:00
4,20261107T132101Z_smoke,3,0,5,0,2024-04-11 20:00:00+00:00
5,20261107T132142Z_dev,3,0,5,0,2024-09-30 23:00:00+00:00
6,20261107T135938Z_dev,3,0,5,0,2024-09-30 23:00:00+00:00
7,20261111T155030Z_smoke,7,0,4,0,2024-04-11 20:00:00+00:00
8,20261111T155739Z_smoke,7,0,8,0,2024-04-11 20:00:00+00:00
9,20261111T160241Z_smoke,4,0,4,0,2024-04-11 20:00:00+00:00


**What it shows.** Every run of the runner so far: 23 runs,
185 step records, 171 forecast files. The test-access log
does not exist, and no step recorded a test access.

**What it means.** 0 forecast hours fall in the test split: the
latest hour any run forecast is before 2024-10-01. The test split is read once, by
the final run (phase 1 forecasts it, phase 2 scores it, D-32), and every read is logged.

## 4. The incident of 2026-11-12 (OP-60) and what it led to

In [5]:
incident = json.loads((OUTPUTS_DIR / "g1_audit" / "incident_2026-11-12"
                      / "regenerated_vs_original_content.json").read_text(encoding="utf-8"))
pd.DataFrame([{"file": k.split("/", 2)[-1], "rows": v["shape_new"][0], "columns": v["shape_new"][1],
               "content equal": v["content_equal"], "writer before": v["created_by_orig"],
               "writer after": v["created_by_new"]} for k, v in incident.items()])

,file,rows,columns,content equal,writer before,writer after
0,04_preprocess/scada_1min_true_utc.parquet,1971360,31,True,parquet-cpp-arrow version 25.0.0,parquet-cpp-arrow version 26.0.0
1,04_preprocess/scada_1H_utc.parquet,32856,35,True,parquet-cpp-arrow version 25.0.0,parquet-cpp-arrow version 26.0.0
2,05_clean/scada_1min_clean.parquet,1971360,48,True,parquet-cpp-arrow version 25.0.0,parquet-cpp-arrow version 26.0.0
3,05_clean/scada_1H_clean.parquet,32856,43,True,parquet-cpp-arrow version 25.0.0,parquet-cpp-arrow version 26.0.0


**What it shows.** Reading the options of the stage modules ran them: stages 4 and 5 were
rewritten on the college machine. The four regenerated tables are compared with the originals,
extracted from the hand-off package and checked against its hashes.

**What it means.** All 4 tables have exactly the original content (158,300,208 cells); only the parquet writer's bytes differ. The
originals were restored, and every entry point now parses its arguments before any work (X62,
T-98).

## 5. The reproduction from the raw data

In [6]:
REPRO = REPO_ROOT / "outputs/g1_audit/reproduction_20261113T202349Z/record.json"
repro = json.loads(REPRO.read_text(encoding="utf-8"))
pd.DataFrame([{"step": s["step"], "exit": s["exit"], "seconds": s["seconds"],
               "files": len(s["compared"]), "failing": len(s["failing"])}
              for s in repro["steps"]])

,step,exit,seconds,files,failing
0,checksums,0,2.6,0,0
1,ingest,0,52.9,4,0
2,clock,0,96.8,8,0
3,preprocess,0,171.2,3,0
4,clean,0,39.9,6,0
5,eda,0,33.1,16,0
6,quality,0,110.7,62,0
7,orientation,0,56.7,3,0
8,features,0,32.5,9,0
9,decide_k,0,42.2,6,3


**What it shows.** 19 steps of a fresh clone, from the checksums of the
raw files to the two development runs of the runner (all six models), each stage's files compared
with this checkout's by content. Result recorded by the script: **NOT reproduced**, because
17 files differ.

**What it means.** 17 of the 17 are explained, each by its cause:
5 are the decision checks of stage 6 and 6b, run on the laptop, which give other
numbers on this machine with the same decisions (X66); 3 are report tables that
differ by the warning that the step now emits (X45), by row order, and by the column `n_resumed` of
the checkpoints (D-38); 9 are learning-curve and trial tables of the TFT models
in which only the column `gpu_peak_mb` differs (X99): the peak of the GPU memory moves with the
timing of Python's garbage collector when any code changes, while the weights stay equal bit for
bit (the probe below). 0 are unexplained. By D-34 this is a consistency check, and
the explained differences are listed and never made to pass by a wider rule.

In [7]:
PROBE = REPO_ROOT / "outputs" / "g1_audit" / "gpu_peak_probe.txt"
print(PROBE.read_text(encoding="utf-8").split("# Output of the probe:")[1].strip())

k=0 (reference)   peak=106.330078125
k=0     peak=108.26416015625        diff vs reference=+1.934082 MB  weights equal=True
k=1     peak=106.33984375           diff vs reference=+0.009766 MB  weights equal=True
k=3     peak=105.92822265625        diff vs reference=-0.401855 MB  weights equal=True
k=17    peak=106.359375             diff vs reference=+0.029297 MB  weights equal=True
k=101   peak=105.94384765625        diff vs reference=-0.386230 MB  weights equal=True
k=700   peak=106.33984375           diff vs reference=+0.009766 MB  weights equal=True
k=5000  peak=105.92822265625        diff vs reference=-0.401855 MB  weights equal=True
gc disabled       peak=106.330078125          diff vs reference=+0.000000 MB  weights equal=True


**What it shows.** After a reference run, the probe trains the same TFT on the same
synthetic windows 8 more times (the last with the cyclic collector switched off);
the only thing that changes between the runs is how many Python container objects are alive before
the fit, which moves the moments at which the collector runs.

**What it means.** The weights are equal bit for bit in all 8 runs, while the peak
of the GPU memory moves from -0.40 to +1.93 MB against the
reference. So the peak is a property of the timing of the host, not of the computation, and a
change of code that creates Python objects (the checkpoints of D-38 do) can move it by the few
kilobytes seen in the reproduction. It was equal in the earlier reproduction, before D-38, and no
decision uses it.

## 6. The two analyses that train models (D-36)

In [8]:
RUN = REPO_ROOT / "outputs/runs/20261113T075845Z_dev"
cv = pd.read_csv(RUN / "cv" / "cv_summary.csv")
display(cv[["target", "horizon", "cv_mean", "cv_std", "validation_uncorrected",
            "validation_minus_cv_pp"]].round(2))
ab = pd.read_csv(RUN / "ablation" / "ablation_summary.csv")
ab[ab["mode"].isin(["no_nwp_gfs", "no_history", "minimal"])][
    ["target", "horizon", "mode", "mean_delta_pp", "margin_pp", "verdict"]].round(2)

,target,horizon,cv_mean,cv_std,validation_uncorrected,validation_minus_cv_pp
0,indirect,1,7.41,2.31,6.79,-0.62
1,direct,1,7.53,2.34,6.66,-0.87
2,indirect,2,7.74,2.38,7.12,-0.62
3,direct,2,7.69,2.10,6.81,-0.88
4,indirect,3,8.08,2.62,6.66,-1.42
5,direct,3,8.11,2.50,6.43,-1.68


,target,horizon,mode,mean_delta_pp,margin_pp,verdict
0,direct,1,minimal,1.26,0.05,helpful
2,direct,1,no_history,0.11,0.05,helpful
5,direct,1,no_nwp_gfs,1.03,0.05,helpful
9,direct,2,minimal,1.21,0.05,helpful
11,direct,2,no_history,0.01,0.05,neutral
14,direct,2,no_nwp_gfs,0.96,0.05,helpful
18,direct,3,minimal,1.52,0.05,helpful
20,direct,3,no_history,0.19,0.05,helpful
23,direct,3,no_nwp_gfs,1.48,0.05,helpful
27,indirect,1,minimal,1.27,0.05,helpful


**What it shows.** S-38: the nRMSE in kW of five expanding date folds of the train rows,
for the XGB models of the development run, against their validation value. S-41: the change in
validation error when a feature group is removed and the model refitted (a subset of modes).

**What it means.** The fold means range from 7.4 to
8.1 %, and the validation value is 0.6 to 1.7 pp
below the CV mean in every case; this is reported and not explained. Removing the GFS forecasts
raises the error by 0.96 to 1.48 pp, the largest change of any mode except
`minimal`, which keeps only the time encodings.
These are single-seed development values and the verdicts are provisional; the final run repeats
them with three seeds and the pooled seed std.

## 7. Checkpoints and resume inside the long steps (D-38, X95)

In [9]:
resume = json.loads((OUTPUTS_DIR / "g1_audit" / "resume_check.json").read_text("utf-8"))
pd.DataFrame([{"configuration": r["name"],
               "first design: largest weight difference": r["first_design"]["max_abs_weight_diff"],
               "final: largest weight difference": r["final"]["max_abs_weight_diff"],
               "dropout between layers": r["interlayer_dropout"]}
              for r in resume["configs"]])

,configuration,first design: largest weight difference,final: largest weight difference,dropout between layers
0,"BiLSTM, 1 layer, dropout 0.1",0.000000,0.0,False
1,"BiLSTM, 2 layers, no dropout",0.000000,0.0,False
2,"BiLSTM, 2 layers, dropout 0.1",0.015304,0.0,True
3,"BiLSTM, 3 layers, dropout 0.3",0.019067,0.0,True
4,"TFT, 1 LSTM layer, dropout 0.1",0.000000,0.0,False
5,"TFT, 2 LSTM layers, dropout 0.1",0.013331,0.0,True
6,"TFT, 3 LSTM layers, dropout 0.3",0.022782,0.0,True


**What it shows.** For each of 7 configurations of the BiLSTM and the
TFT, a fit of 5 epochs is continued from the checkpoint after epoch
2, and its weights are compared with those of the uninterrupted fit
(`scripts/g1_resume_check.py`, synthetic windows, on GRID P40-24Q). The first design restored every
random state of torch; the final one also runs the earlier training passes again.

**What it means.** The final design is equal to the uninterrupted fit in
7 of 7 configurations. The
first design failed in the 4 that have dropout between their LSTM layers (weights up
to 2.3e-02 apart): cuDNN keeps a
dropout state that torch's random state does not hold (X95). The replay costs
15 to 34 % of the seconds of the epochs it
replaces. A worker that is killed and started again, and the xgb step, continue to the files of an
uninterrupted run (T-116, T-118); this table is the check of the fit itself.

## 8. Where the gate stands

The checklist, with the state of every item read from its evidence file, is figure S09-34 of the
figure book (`docs/figure_sheets/S09-W6.md`). What remains is outside this notebook: the
external review, the user's approval of the tag, the tag `g1-frozen`, and the final run by
`docs/runbook_final.md` in a plain PowerShell window of the Horizon desktop.